In [ ]:
# 09/15
import nltk

def ngrams(sentence, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)
sentence = '안녕하세요. 만나서 진심으로 반가워요'

unigram = ngrams(sentence,1)
bigram = ngrams(sentence,2)
trigram = ngrams(sentence,3)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))

[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요')]


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = ['That movie is famous movie',
          'I like that actor',
          "I don't like that actor"]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [5]:
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size,
                 embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                                      embedding_dim=embedding_dim)
        self.linear = nn.Linear(in_features=embedding_dim,
                                out_features=vocab_size)
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [6]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"

import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS10\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS

In [7]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [8]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens,
                    n_vocab=5000,
                    special_tokens=['<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [9]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx-window_size)
            window_end = min(sentence_length,
                             idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx + 1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs
word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [10]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id['<unk>']
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:,1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset,
                        batch_size=32,
                        shuffle=True)

In [12]:
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
word2vec = VanillaSkipgram(vocab_size=len(token_to_id),
                           embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

cuda

Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [13]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss.item()
    cost = cost / len(dataloader)
    print(epoch + 1, cost)

1 6.197169351845234
2 5.980928698818097
3 5.931315908327609
4 5.901037595218334
5 5.878887838294905
6 5.861362090857354
7 5.8466148416232695
8 5.8337806612032255
9 5.82229190013785
10 5.811973379398154


In [14]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding
index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-2.5300738e-01  1.5244348e+00  2.1306981e-01  4.8298959e-02
 -4.8304430e-01  6.6923833e-01 -3.3756030e-01  5.2835584e-01
  6.0382593e-02 -1.5941282e-03  5.6892610e-01 -2.6057813e-01
 -1.3045937e+00  1.5365936e+00  2.5463245e+00  3.9136320e-01
  1.6815075e-01  3.8616166e-01  7.5933617e-01 -9.6385133e-01
  6.7984635e-01 -8.4964013e-01  2.7149066e-01  4.8960191e-01
 -1.7902965e-02 -5.4557627e-01  1.6752569e-01  9.9337894e-01
  3.2730749e-01  9.8434645e-01 -7.2396231e-01  2.1016303e-01
  5.2187777e-01 -9.0146858e-01 -2.4388722e-01 -7.2005337e-01
 -2.3540314e-01  1.1626719e+00  4.2163312e-02  3.6001271e-03
  5.5170304e-01  6.5392989e-01 -1.3803775e+00 -2.3136809e-01
  1.2553756e-01  4.5575052e-03  2.0286672e+00 -2.8037045e-02
 -3.9502156e-01 -6.7788899e-01 -2.4133651e-01 -2.5106199e+00
 -9.7853494e-01  9.7590959e-01  1.8113490e+00 -1.2815492e-01
 -3.0215809e-01 -7.4836606e-01 -1.4043820e+00 -2.9083425e-01
 -1.8839540e-01  9.5368460e-02  9.6093774e-01 -2.5052643e-01
 -2.1289017e+00 -1.50

In [15]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1:n+1]
    return top_n
cosine_matrix = cosine_similarity(token_embedding,
                                  embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

for index in top_n:
    print(id_to_token[index], cosine_matrix[index])

코스트 0.33565995
할말이 0.29798657
연기력 0.29610538
여름 0.2922611
KBS 0.2848858


In [ ]:
# 09/16
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS10\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS

In [20]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]

In [21]:
from gensim.models import Word2Vec

word2vec = Word2Vec(sentences=tokens,
                    vector_size=128,
                    window=5,
                    min_count=1,
                    sg=1,
                    epochs=3,
                    max_final_vocab=10000)

In [22]:
word2vec.save('./models/word2vec.model')
word2vec = Word2Vec.load('./models/word2vec.model')

In [23]:
word = '연기'
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2='연기력'))

[-0.48926622 -0.11271544  0.22113225  0.37312695 -0.1666797  -0.1523364
  0.09830394  0.14005221 -0.6083869   0.4761507   0.08519541 -0.4156145
 -0.12251072 -0.09325932  0.08360821 -0.0442     -0.13864343  0.09123334
 -0.00635281  0.1673483   0.49085057  0.28430563 -0.13532443 -0.02315556
 -0.50339895 -0.11227059 -0.40218496  0.13827777  0.3159315  -0.1603719
 -0.5448027   0.02332031  0.25946927 -0.17494103 -0.10923038 -0.02387715
  0.12728074  0.03177891 -0.09161357 -0.22139464 -0.12037103  0.21052031
 -0.20489526 -0.47074032 -0.30853578  0.4446949  -0.47816637 -0.11881668
  0.36830094 -0.12911052  0.59615207  0.24424843  0.13653205  0.25768533
 -0.30607945 -0.42117295 -0.00473919  0.05830995 -0.21226567 -0.10297433
 -0.02174789 -0.10208131  0.05902335  0.19174609 -0.24399656  0.24701649
 -0.17808458  0.4020758   0.2551767  -0.34463078 -0.37883928 -0.5221825
 -0.32325828  0.0425496  -0.01696942 -0.0555882   0.02167227 -0.25265524
 -0.16458456  0.01742364 -0.06893975 -0.16340466  0.332

In [24]:
from Korpora import Korpora

corpus = Korpora.load('kornli')
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

[kornli] download multinli.train.ko.tsv: 83.6MB [00:01, 77.1MB/s]                            
[kornli] download snli_1.0_train.ko.tsv: 78.5MB [00:00, 88.0MB/s]                            
[kornli] download xnli.dev.ko.tsv: 516kB [00:00, 7.05MB/s]
[kornli] download xnli.test.ko.tsv: 1.04MB [00:00, 14.4MB/s]


[['개념적으로', '크림', '스키밍은', '제품과', '지리라는', '두', '가지', '기본', '차원을', '가지고', '있다.'], ['시즌', '중에', '알고', '있는', '거', '알아?', '네', '레벨에서', '다음', '레벨로', '잃어버리는', '거야', '브레이브스가', '모팀을', '떠올리기로', '결정하면', '브레이브스가', '트리플', 'A에서', '한', '남자를', '떠올리기로', '결정하면', '더블', 'A가', '그를', '대신하러', '올라가고', 'A', '한', '명이', '그를', '대신하러', '올라간다.'], ['우리', '번호', '중', '하나가', '당신의', '지시를', '세밀하게', '수행할', '것이다.']]


In [25]:
from gensim.models import FastText

fastText = FastText(sentences=tokens,
                    vector_size=128,
                    window=5,
                    min_count=5,
                    sg=1,
                    max_final_vocab=20000,
                    epochs=3,
                    min_n=2,
                    max_n=6)

In [26]:
oov_token = '사랑해요'
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn=5))

False
[('사랑', 0.8738827109336853), ('사랑에', 0.8183375597000122), ('사랑의', 0.7849916815757751), ('사랑을', 0.7520293593406677), ('사랑하는', 0.7444128394126892)]


In [32]:
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [34]:
input_size = 128
output_size = 256
num_layers = 5
bidirectional = True

model = nn.RNN(input_size=input_size,
               hidden_size=output_size,
               num_layers=num_layers,
               nonlinearity='tanh',
               batch_first=True,
               bidirectional=bidirectional).to(device)
batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size,
                     sequence_len,
                     input_size).to(device)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 output_size).to(device)
outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print(hidden.shape)
print(outputs.device)

torch.Size([4, 6, 512])
torch.Size([10, 4, 256])
cuda:0


In [36]:
input_size = 128
output_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(input_size=input_size,
                hidden_size=output_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                proj_size=proj_size)
batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size,
                     sequence_len,
                     input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 proj_size if proj_size > 0 else output_size)
c_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 output_size)
outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


In [ ]:
import torch
import torch.nn as nn

class SentenceClassifier(nn.Module):
    def __init__(self,
                 n_vocab,
                 hidden_dim,
                 embedding_dim,
                 n_layers,
                 dropout=0.5,
                 bidirectional=True,
                 model_type='lstm',
                 pretrained_embedding=None):
        super().__init__()
        if pretrained_embedding is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(pretrained_embedding, dtype=torch.float32))
        else:
            self.embedding = nn.Embedding(num_embeddings=n_vocab,
                                          embedding_dim=embedding_dim,
                                          padding_idx=0)
        if model_type == 'rnn':
            self.model = nn.RNN(input_size=embedding_dim,
                                hidden_size=hidden_dim,
                                num_layers=n_layers,
                                bidirectional=bidirectional,
                                dropout=dropout,
                                batch_first=True)
        elif model_type == 'lstm':
            self.model = nn.LSTM(input_size=embedding_dim,
                                 hidden_size=hidden_dim,
                                 num_layers=n_layers,
                                 bidirectional=bidirectional,
                                 dropout=dropout,
                                 batch_first=True)
        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(dropout)
    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

In [38]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
corpus_df = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS10\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS

In [41]:
train = corpus_df.sample(frac=0.9,
                         random_state=42)
test = corpus_df.drop(train.index)

print(train.head().to_markdown())
print(len(train))
print(len(test))

|       | text                                                                                     |   label |
|------:|:-----------------------------------------------------------------------------------------|--------:|
| 33553 | 모든 편견을 날려 버리는 가슴 따뜻한 영화. 로버트 드 니로, 필립 세이모어 호프만 영원하라. |       1 |
|  9427 | 무한 리메이크의 소재. 감독의 역량은 항상 그 자리에...                                    |       0 |
|   199 | 신날 것 없는 애니.                                                                       |       0 |
| 12447 | 잔잔 격동                                                                                |       1 |
| 39489 | 오랜만에 찾은 주말의 명화의 보석                                                         |       1 |
45000
5000


In [45]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus,
                n_vocab,
                special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab
tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens,
                    n_vocab=5000,
                    special_tokens=['<pad>', '<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

In [46]:
import numpy as np

def pad_sequences(sequences,
                 max_length,
                 pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)
    return np.asarray(result)
unk_id = token_to_id['<unk>']
train_ids = [[token_to_id.get(token, unk_id) for token in review] for review in train_tokens]
test_ids = [[token_to_id.get(token, unk_id) for token in review] for review in test_tokens]

max_length = 32
pad_id = token_to_id['<pad>']
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [48]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values,
                            dtype=torch.float32)
test_labels = torch.tensor(test.label.values,
                           dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset,
                          batch_size=16,
                          shuffle=True)
test_loader = DataLoader(test_dataset,
                         batch_size=16,
                         shuffle=False)

print(train_loader)
print(test_loader)

C:\Users\KDS10\AppData\Local\Temp\ipykernel_28912\253373603.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_ids = torch.tensor(train_ids)
C:\Users\KDS10\AppData\Local\Temp\ipykernel_28912\253373603.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_ids = torch.tensor(test_ids)


In [53]:
import torch.optim as optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

classifier = SentenceClassifier(n_vocab=n_vocab,
                                hidden_dim=hidden_dim,
                                embedding_dim=embedding_dim,
                                n_layers=n_layers).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.0001)

In [54]:
def train(model, datasets,
          criterion, optimizer,
          device, interval):
    model.train()
    losses = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval == 0:
            print(f"Train Loss {step} : {np.mean(losses)}")


def test(model, datasets, criterion, device):
    model.eval()
    losses = list()
    corrects = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits)>.5
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )

    print(np.mean(losses), np.mean(corrects))

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier,
          train_loader,
          criterion,
          optimizer,
          device,
          interval)
    test(classifier,
         test_loader,
         criterion,
         device)

Train Loss 0 : 0.6920034289360046
Train Loss 500 : 0.6936473167109156
Train Loss 1000 : 0.6932125051657517
Train Loss 1500 : 0.6930192775681843
Train Loss 2000 : 0.6930160234118627
Train Loss 2500 : 0.6929656310779292
0.6926394472487818 0.5318
Train Loss 0 : 0.6924538612365723
Train Loss 500 : 0.6922063525327428
Train Loss 1000 : 0.6848213234386006
Train Loss 1500 : 0.667364581396864
Train Loss 2000 : 0.6521698818511811
Train Loss 2500 : 0.6386754748250236
0.5637517914223594 0.7122
Train Loss 0 : 0.5513736009597778
Train Loss 500 : 0.5446333804292355
Train Loss 1000 : 0.5390253223322489
Train Loss 1500 : 0.5336027162461023
Train Loss 2000 : 0.525970047672113
Train Loss 2500 : 0.5201529486269438
0.48959691065568894 0.7572
Train Loss 0 : 0.6125587821006775
Train Loss 500 : 0.4711073544508445
Train Loss 1000 : 0.46512444439408307
Train Loss 1500 : 0.46273230511613245
Train Loss 2000 : 0.4613215587605005
Train Loss 2500 : 0.4598891571289251
0.45204240712114035 0.7842
Train Loss 0 : 0.38494

In [55]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
    token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

보고싶다 [ 0.26168212 -1.9158787  -0.257827    0.31930974 -0.98265755  0.48262227
  0.9612633   0.6823222  -2.18329    -0.8211915   0.32266578  2.151314
 -0.03842838  0.12415574  0.47101972  0.52789134 -0.14733367 -0.4448138
 -0.12351849 -0.23873055  0.14756931  1.0458752  -2.19083    -0.7603365
  0.88862216 -0.9221471  -0.27348924  0.61155057 -0.05021084  1.2794137
 -0.9886313  -0.19918616  1.3442751   0.7162358  -0.61558753  0.07714098
  0.5042257   0.46269882 -0.40788314 -0.225371   -0.803422    2.1236625
 -0.06047428 -1.5636797   0.5632328  -1.135242   -0.1628568  -0.865597
  0.59906536 -1.541307   -1.1657267  -2.669707    1.7266309  -0.78842646
  0.3328597  -2.483521    2.1688168  -2.858809    1.0877872   0.0698043
 -0.18515083 -0.9629711   0.20642802 -1.4557261  -1.237805   -0.8089879
  1.1988733  -1.366318    0.11042535 -1.2253405   0.60618454 -0.7173493
 -1.8800586  -0.83155763  1.9481889  -1.1965654   0.10180673 -0.34860754
  0.20163283 -0.18786931 -0.07338709  1.9855326   0.35971